## **In Depth Review: Highest Offense Calculator**

### **Conceptual Design**
The *Highest Offense Calculator* is an object-oriented programming style, designed to provide a practical solution when trying to extract the most serious offense within a case. The programming script has three main features, slicing columns, backfill function, and the calculator itself. The slicer is a standardizing method to ensure consistency of code column throughout any dataset. Next, the backfill feature is an integral part of accuracy, which fills in missing data points and has capabilities in exception handling. Lastly, the calculator itself, which is designed to fit any dataset. Afterwords, all files can be exported as a CSV file. 

##### *Design Comparison to Previous Attempts*
There are only two other previous attempts of optimizing and improving highest crime calculator. The biggest differences between previous attempts and this one are the ability to fit all datasets & simplicity of use, exception handling, calculator logic, and the offense codes data. 

In terms of fitting all datasets, previous attempts required the entirety of the code and necessity to change each section of the script requiring manual fill in of column names. This becomes an un-necessarily and time consuming step, and can pose issues when python code cannot be read well enough for users to know which columns are in what input. With this attempt, the dataset and key columns are specified as part of the initialization of the data, allowing data extraction to be as error friendly as possible. The simplicity of the new calculator is that the back frame of the code itself is not seen. Instead, it is a library import where users can utilize the functions in it. 

With exception handling, this script has several areas where it is necessary to exclude or include certain values. In terms of the calculator, the two most important areas of handling exceptions are the standardizing of codes and filling offense class descriptions. When standardizing the column of offense codes, some datasets had ending digits. These digits correlated to the statute section's items or clauses. The ending digits caused issues between 900 codes (homicides) and 9,000 codes (miscellaneous). When executing previous attempts, if a dataset had ending digits, the calculator would classify all high serious offenses as low-level miscellaneous offenses. With the improved function for standardizing codes, the script knows whether to extract the first 3 or 4 digits of the code, based on the pattern found. The other exception handling issue was with the backfill function. Stated further down, in the *Input Data* section, there are varying values that sometimes hold the case degree. With previous attempts, the column was wiped and merged with the *All_OffenseCodes.csv* file provided by the state. This meant that any class specifications would be re-classified and no longer specific to the case or the original data. 

Regarding calculator logic, previous calculators re-classify and rank the classes. Based on this, alone, if a case were to have two Felony A charges, the algorithms would choose at random which offense to keep as the highest charge. Therefore, at random, two Felony A charges, where one charge may have the higher code number could be chosen to be the highest offense. With this version, the baseline of only finding the highest offense of a case, without the argument of finding a specific class, the calculator is solely based on the code itself as it is already a ranking system given by the State. If there were to be several counts of the highest offense, it would not be affected as they are both the same charge. 

The offense codes data in this calculator comparative to others is that it was manually edited. Many offense degree values were missing due to various factors, such as being a 'place holder charge' or being a common law. Due to this, all 'blank', '??', and missing values were filled in. This drastically increased accuracy of extraction of specific class, as the formatting of offenses with several outcomes has been accounted for in the python script and in the file itself. More details are in the *Input Data* section.

##### *Limitations*
There were several limitations to previous attempts, which were improved by this third attempt. However, there are still two limitations within this version. (1) Cleaning data, while this calculator perfectly maintains current data points and fills in missing data points, it still requires manual cleaning of each column. This being an easy fix utilizing statistical software or data visualization platforms. (2) The offense codes data is an incurable limitation, because if an offense code does not exist in this file, then it cannot backfill missing data within the files needed for calculating. As a result, when finding the highest crime with the stipulation of a specific class, this version will automatically exclude anything without a value in the class column.

### **Table of Contents**
1. CJ_Services Library Overview
    - Library Imports
    - Constructor
    - Get Data Method
    - Slice Column Code Method
    - Fill Blank Column Method
    - Export File Method
    - Calculate Highest Offense Method
2. Input Data
    - Offense Codes File
    - Test Files

3. Demonstration
    - Demo 1: Simplistic Code Review
    - Demo 2: Library Walk Through & Format Options
        - (1) imports
        - (2) initialize offense file
        - (3) initialize cjs.DataSource()
        - (4A) backfill class column
        - (4B) backfill other columns
        - (5A) calculate highest offense
        - (5B) viewing cjs.DataSource() in python file
        - (6) export dataset to CSV
    - Demo 3: Nested Results Comparison of data after each step

### **Class Overview `CJ_Services`**
##### *Library Imports*
Four library imports are utilized and must be initiated at the beginning of the programming script: `pandas`, `numpy`, `re`, and `typing`. *Pandas* library is an industry standard data analysis tool, that assists in easy reading, filtering, and exporting files. *Numpy* library is another industry standard tool that is utilized for complex aggregation, scalar functionality, and analytical observations. In this script, numpy is used for string manipulation and exception handling. Regular Expression (*re*) is a special function for pattern searching, making sure there are no collisions when searching for a specific letter, word, or phrase. Lastly, *typing* is a specialized library which allows for specified functions to have optional attribute handling.

In [1]:
import pandas as pd
import numpy as np
import re
from typing import Optional, Literal, Union, List

##### *Constructor*
The first section and most important aspect of object-oriented programming is the constructor. It consists of the class name, the constructor variable, and constructor attributes. This is the key to making the calculator fit a variety of datasets as it stores where the file can be found, and specific columns needed for execution. These columns are the file itself, primary key, the class, and the code. When initializing the dataset, it is optional to initialize either a `pandas.DataFrame()` or a file path. In doing this, it allows for pre- or post-analysis utilizing python. The primary key is how the calculator should group the offenses and most commonly is by case number. However other columns can be substituted and such as PID number. After initializing the constructor, the dataset will automatically be stripped of leading and ending spaces, and assigned to the private variable `_data`. As common practice, a second private variable called `_newData` will store the new dataset. Having separate variables allows for easy reversion of data extraction and future use. Lastly, an empty variable called `_droppedRows` will be created to store any rows that have been dropped from the `cjs.Fill_BlankCol()` method. This is more or a less a check method and will only be needed when needing to see the invalid values in the `_Col_Code` or if a high percentage of rows were dropped. 

**Note:** Primary Key column does not have to be specified and will not prevent the calculator method from functioning.

In [2]:
class DataSource:
    def __init__(self, *, CSV_File: Union[str, pd.DataFrame], Col_PrimKey: Optional[str] = None, Col_Class: str, Col_Code: str):
        # Initializing constructor variables
        self._CSV_File = CSV_File
        
        # Col_PrimKey Attribute Check
        if Col_PrimKey is not None:
            self._Col_PrimKey = Col_PrimKey
        self._Col_Class = Col_Class
        self._Col_Code = Col_Code
        
        # CSV_File instance check for specific pandas initialization
        if isinstance(CSV_File, str):
            initial_df = pd.read_csv(CSV_File, low_memory=False)
        else: initial_df = CSV_File
        
        # Variables for stripping and storing dataset
        self._data = initial_df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
        self._newData = None
        
        # Variable for storing dropped rows from cjs.Fill_BlankCol()
        self._droppedRows = None

##### *Get Data Method* `cjs.Get_Data()`
This function allows for easy access to the original dataset and the dataset after code execution. This is in place because the `DataSource` class will become a library import. Meaning that access to the raw data stored within the constructor will not be accessible unless a specialized function is created to do so. This will allow for further analysis within python and `pandas.DataFrame()` imports. The `Get_Data` function has one optional parameter that allows users to specify if they want the 'calculated' data or the 'original' data. If there is no specification, the automatic return will be the most recent or updated dataset. To prevent any confusion, statements specifying which dataset has been returned will be printed for users. At the same time, if a specific dataset is called but does not exist, the function will return `None` and print a statement of the corresponding to the data structure selected. In the case that a dataset does not exist in either the `_data` or `_newData` variable, the function will implicitly return nothing.

In [ ]:
def Get_Data(self, Data: Optional[Literal["calculated data", "original data"]] = None):
    # Execution if attribute is specified
    if Data == "calculated data":
        if self._newData is not None:
            print("Returning Calculated Data")
            return self._newData
        print("Data has not been calculated")
        return
    elif Data == "original data":
        if self._data is not None:
            print("Returning Original Data")
            return self._data
        print("Data has not been intialized")
        return

    # Execution if parameter is not specified
    elif Data is None:
        if self._newData is not None:
            print("Returning Calculated Dataset")
            return self._newData
        elif self._data is not None:
            print("Returning Original Dataset")
            return self._data
            
        # Execution if function has no dataset
        print("No Existing Dataset")
        return

##### *Slice Column Code Method* `cjs.SliceCol_Code`
The `cjs.SliceCol_Code()` is a supporting method to standardize and normalize the column that contains all the offense codes. This allows for consistency and easy matching when back filling data and for analyzing the most serious offense. This method results in creating a new column called `Code (4)`, stripping any existing leading zeroes, exception handling, and extract either the first 3 digits or 4 digits of the code column. Based on the *All_OffenseCodes.csv* file, the crimes involving a homicide are coded from *910* to *998*, but the low-level miscellaneous crimes are coded in the *9,900* to *9,999*. Without specification of which code should be extracted as 3 or 4 digits, the script will miscalculate homicides as low-level offenses. Thus, requiring exception handling, which is best solved by using `numpy.select()` method. Lastly, the new column, `Code (4)` will be assigned in memory as the new private variable `_Col_Code`, originally initialized in the constructor.

**Note**: For free text of *999* (homicide) and *9,999* (other crime), cannot be accounted for. Therefore, if there were to be a free text for homicide, it would be normalized as *9,999*, causing it to be considered the lowest crime. Currently, this is not too much of a concern, given that free text is rare in the data warehouse.

In [4]:
def SliceCol_Code(self):
    # Create new column
    newCol_Name = 'Code (4)'
    col_code = self._data[self._Col_Code]
        
    # Standardize codes
    cleaned_codes = col_code.astype(str).fillna('')
    cleaned_codes = cleaned_codes.str.lstrip('0')
        
    # Exception Handling
    prefix_3digit = ('91', '92', '93', '94', '95', '96', '97', '98')
    condition_3digit = cleaned_codes.str.startswith(prefix_3digit)
    choice_3digit = cleaned_codes.str[:3]
    choice_4digit = cleaned_codes.str[:4]
    self._data[newCol_Name] = np.select(
        condlist=[condition_3digit],
        choicelist=[choice_3digit],
        default=choice_4digit
    )
        
    # Re-Assignment of Column Code
    self._Col_Code = newCol_Name
    self._data[newCol_Name] = pd.to_numeric(self._data[newCol_Name], errors='coerce')

##### *Fill in Blank Column Method* `cjs.Fill_BlankCol()`
The `cjs.Fill_BlankCol()` function is another supporter method designed to handle null columns in regards to offense. These columns could be the offense type, name, degree, or general statute. The automatic assumption of this method is to fill in the `_Col_Class`. Although, the design of this function is versatile to any column that is specified. If necessary, the function will need to be ran several times if there are several columns that have significant null values. Lastly, it is designed to handle exceptions, if necessary. Referring to the [Test_Data.csv](#test_data.csv), `Offence Type Desc` column has partial information where some values represent only the type while others represent the type and degree. The `Value_Exception=` handling for the `cjs.Fill_BlankCol()` is to keep the values that have the degree in them, while filling in the ones that have no pertinent information. This parameter is not case sensitive to capitalization but is sensitive to spelling. Most importantly, the function needs the [All_OffenseCodes.csv](#all_offensecodes.csv) to be specified as an attribute, which will need to be initialized outside of the function.

**Note 1**: This method is not meant to standardize or normalize the columns that are ran through, therefore manual cleaning is necessary. However, even with an un-standardized column, the calculator will still function normally. 

**Note 2**: For any row with missing codes, the row will be dropped completely. This is also not too concerning as many observed rows that has a null value for offense code also has a null value for the rest of the row. A statement will print how many rows were dropped in the process.

**Note 3**: There are duplicate codes due to a change in general statutes. This means that the second instance of the code in the *All_OffenseCodes.csv* is the most up to date and will be the one that is kept, when dropping one of the duplicate codes.

In [ ]:
def Fill_BlankCol(self, *,
                  File_OffenseCodes: pd.DataFrame,
                  File_colNull: Optional[str] = None,
                  Data_colNull: Optional[str] = None,
                  Value_Exception: Optional[Union[str, List[str]]] = None):
        
    # Column Code (4) Check
    if 'Code (4)' not in self._data.columns:
        self.SliceCol_Code()
            
    # Stripping spaces from Offense_Codes.csv
    File_OffenseCodes = File_OffenseCodes.apply(lambda x: x.astype(str).str.strip())
        
    # File_ColNull Optional Parameter Check: automatic assumption is filling in class column
    if File_colNull is None:
        File_colNull = 'CL'
            
    # Subsetting & Copying specified columns for merge
    lookup_data = File_OffenseCodes[['CODE', File_colNull]].copy()
    lookup_data = lookup_data.drop_duplicates(subset=['CODE'], keep='last')
    lookup_data['CODE'] = pd.to_numeric(lookup_data['CODE'], errors='coerce')
        
    # Data_colNull Optional Parameter Check: automatic assumption is filling in class column
    if Data_colNull is None:
        Data_colNull = self._Col_Class
            
    # Initialzie a new column to be the replacement of old column
    temp_col = f'{self._Col_Class}_original'
        
    # Value_Exception Optional Attribute Check: automatic assumption is there are no exceptions
    if Value_Exception is not None:
        if isinstance(Value_Exception, list):
            Value_Exception = [item.upper() for item in Value_Exception]
        else:
            Value_Exception = Value_Exception.upper()
        Value_Exception = [item.upper() for item in Value_Exception]
        self._data[Data_colNull] = self._data[Data_colNull].str.upper()
        self._data[Data_colNull] = self._data[Data_colNull].replace(to_replace=Value_Exception, value=np.nan)
    self._data.rename(columns={Data_colNull: temp_col}, inplace=True)
        
    # Counting rows before and after null drop
    rows_b4_drop = len(self._data)
    self._droppedRows = self._data[self._data[self._Col_Code].isna()]
    rows_after_drop = self._data.loc[self._data[self._Col_Code].notna()].copy()
    print(f"Dropped {rows_b4_drop - len(rows_after_drop)} rows where '{self._Col_Code}' was missing.")
        
    # Merging Offense_Codes with Dataset temporary column
    merged_df = pd.merge(
        rows_after_drop,
        lookup_data,
        left_on=self._Col_Code,
        right_on='CODE',
        how='left'
    )
        
    # Filling in original column nulls with the temporary column
    merged_df[Data_colNull] = merged_df[temp_col].fillna(merged_df[File_colNull])
        
    # Dropping all columns created by Fill_BlankCol method
    merged_df.drop(columns=[temp_col, 'CODE', File_colNull], inplace=True, errors='ignore')
    self._data = merged_df

##### *Get Dropped Method* `cjs.Get_Dropped()`
This is a checking function where any rows that were dropped from the `cjs.Fill_BlankCol()` will be stored in this method. In doing this, it allows for analysts to check what the invalid values within the `_Col_Codes` are. Rows in the `cjs.Calc_HighestOffense()` will not be stored anywhere due to it not being necessary. This is a check function, which will not be needed in most cases.

**Note:** Demonstration use of this method will be in Demo 2 and 3 only.
**Note:** Since the original column will be dropped, a new column name will be assigned with the added "_original" to the name.

In [6]:
def Get_Dropped(self):
    if self._droppedRows is None:
        print("No values were dropped")
    return self._droppedRows

##### *Export File Method* `cjs.Export_File()`
This is a supporter method for easy exporting of any file utilized in this python script. The standard is that all files will be exported as a CSV, and the exported file will only be the latest version of it. Within this script, the file will export into a specified folder by the user, based on `Folder_Path=` parameter. However, if no folder path is stated, the file will export to the *Main Page* where the python execution is. The name of the exported dataset will have the same name as the original file. Although, if the `cjs.DataSource()` was initialized using a `pandas.DataFrame()` variable, then it's return name will be 'pd_DataFrame'. If needed, the name of the exported file can be changed by filling in the parameter `File_Name=`, but this is optional and will not affect function if nothing is written. Afterwords, a statement will print out as to the name and file location. In most cases, the name of the file will be one of the following.
- PROCESSED_[Original File Name].csv
- _PROCESSED_[Original File Name].csv
- PROCESSED_[New Fil Name].csv
- _PROCESSED_[New File Name].csv
- PROCESSED_pd_DataFrame.csv
- _PROCESSED_pd_DataFrame.csv

**Note:** necessary folder formatting of folder names needs to be known prior to execution.
- ..[Folder Name]/ -> folder is nested outside of folder which python script is running
- [Folder Name]/ -> folder is nested inside of folder which python script is running

In [7]:
def Export_File(self, Folder_Path: Optional[str] = None, File_Name: Optional[str] = None):
        
    # Version Check
    if self._newData is not None:
        df_to_export = self._newData
    else:
        df_to_export = self._data
    
    # File Name Check or Name Extraction
    if File_Name is None:
        if isinstance(self._CSV_File, str):
            folder, file_path = self._CSV_File.rsplit('/', 1)
            File_Name = file_path.rsplit('.', 1)[0]
        else:
            File_Name = 'pd_DataFrame'
    
    # Drop rows where only all values are missing
    df_to_export.dropna(how='all', inplace=True)
    
    # Separation of original file and exported file
    df_to_export = df_to_export.copy()
        
    # Replacing null value representation
    df_to_export = df_to_export.replace(np.nan, '')
        
    # Standardizing all values to str for easy Excel file reading
    df_to_export = df_to_export.astype(str)
        
    # Folder_Path Attribute Check & Format Check
    if Folder_Path is None:
        Folder_Path = '_'
    elif Folder_Path is not None:
        if Folder_Path[-1] != '/':
            Folder_Path = Folder_Path + '/'
    
    # Export File
    df_to_export.to_csv(
        f"{Folder_Path}PROCESSED_{File_Name}.csv", 
        index=False, 
        encoding='utf-8'
    )
    
    # Print Statement of File Name and Location
    if Folder_Path == '_':
        print(f'Exported as "{Folder_Path}PROCESSED_{File_Name}.csv"')
        return print()
    print(f'Exported in "{Folder_Path}" folder as "PROCESSED_{File_Name}.csv"')
    return print()

##### *Calculate Highest Offense Method* `cjs.Calc_HighestOffense()`
The main function of this library is the `cjs.Calc_HighestOffense()`, which groups the dataset based off of a specific column specified in the `Group_By` parameter. Common columns to group by could be by person ID, case number, or booking ID. If there is no specification for how the offenses should be grouped, the automatic assumption is based on the `_Col_PrimKey` constructor attribute. Although, if a column other than the primary key is to be the main grouper or if the primary key was never written in the constructor, the `Group_By=` attribute must be stated. The calculator first standardizes the codes in the given dataset, then groups the offenses, and lastly finds the smallest value of the offense codes. This being that NC Courts made the numbers correlate with the seriousness, based on lowest values being the most serious. Since the calculator utilizes only `Code (4)` column, there is a high potential of the caluclator returning 'Blank', '??', or null values for the degree type. This is due to the fact that the information provided in the *All_OffenseCodes.csv* has missing data other than the offense code. This issue will mostly occur within Traffic and Infraction cases. Other than that, if specified in the `Find_Class=` parameter, the calculator can also find the highest crime of a specific class. This parameter is not case sensitive. With this, any 'Blank', '??', or null values for the degree type will be filtured out. When utilizing `cjs.Calc_HighestOffense()` method, the aggregated data will be nested into the `_newData` variable. This is both common practice, and the best method in securing the original data, without deleting any rows. Easy extraction of either original or aggregated data can be processed through the `cjs.Get_Data()` function.

**Note 1**: Prior to data pull in MSSQL, raw data must be used for calculator to work accurately.

**Note 2**: In the constructor, if `Col_PrimKey` was not specified, the `Group_By=` parameter is mandatory.

**Note 3**: When finding the highest crime of a specific class, any row that has '??', 'Blank', or is Null will automatically be dismissed. Otherwise, the calculator funcitons on the `Code (4)' column.

In [ ]:
def Calc_HighestOffense(self, 
                        Group_By: Optional[str] = None, 
                        Find_Class: Optional[str] = None):
        
    # Check Code (4) Column
    if 'Code (4)' not in self._data.columns:
        self.SliceCol_Code()
            
    # Group_By Parameter Check: automatic assumption is Primary Key Column
    if Group_By is None:
        Group_By = self._Col_PrimKey
            
    # Find indexes of the smallest CODE values of each group
    indexes_minCode = self._data.groupby(Group_By)[self._Col_Code].idxmin()
    
    # Initialize in new data, securing old data
    newData = self._data.loc[indexes_minCode]
    self._newData = newData
   
    # Find_Class Parameter Check: is not an automatic function
    if Find_Class is not None:
            
        # Prevents case sensitive formatting
        Find_Class = Find_Class.upper()
        search_pattern = r'\b' + re.escape(Find_Class) + r'\b'
        if self._newData is not None:
            self._newData[self._Col_Class] = self._newData[self._Col_Class].astype(str).str.upper()
                
            # Subsetting newData based on if it contians the specified class
            newData = self._newData[self._newData[self._Col_Class].str.contains(search_pattern, regex=True, na=False)]
            self._newData = newData

### **Input Data**
##### *All_OffenseCodes.csv*
Starting with the `offense_codes` data, this document is provided by the [NC Courts](https://www.nccourts.gov/documents/publications/nc-courts-offense-codes-and-classes) containing all the current abbreviations, offenses, and their codes. Without modification, this document cannot be utilized in the calculator, as missing data needs to be addressed prior to python script execution. The missing data is referring to the `CL` column, where the classes of certain offenses are labeled as '??' or is a blank value. With the missing values, the calculator excludes these variables, leading to a high potential of miscalculation in data extraction. With the complexity of sentencing, a high number of offenses have several class outcomes, depending on the stipulation of the person and their alleged crime. In this demonstration, all missing or '??' values in the `CL` column were filled in manually based on their general statutes. The improved offense codes results in any 'place holder charge' or blank charges to have several classes as their class description. In doing this, it accounts for all possibilities of offense class and prevents over exclusion of rows when needing to find the highest offense of a specified class. The formatting of the new offense code's file is demonstrated below with murder charges. Below are variations of the class column. There are still 'Blank' values due to there not being any class stipulation in the general statute, or there being no statute found. 

**Note**: Without an improved version of the offense codes, any version of the highest offense calculator will exclude missing, 'Blank', and '??' values. This being the root cause of the miscalculation when trying to find the highest offense in a specified class.

In [9]:
import pandas as pd
offense_codes = pd.read_csv('TestCSV_FilesInput/Test_OffenseCodes.csv')
file_rExample_rException = offense_codes.iloc[[0, 9, 23, 30]][['CODE', 'Offense Description', 'CL']]
file_rExample_rException

,CODE,Offense Description,CL
0,910,MURDER OF AN UNBORN CHILD,A
9,930,MURDER,A/B1/B2
23,955,SOLICITATION TO COMMIT MURDER,Blank
30,999,HOMICIDE - FREE TEXT,??


##### *Test_Data.csv*
For the demonstration, the `test_data` is an old query performed in data warehouse. This spreadsheet illustrates the key components and randomness of datasets created from the warehouse. Often, most of the information related to the offense is missing, which includes the offense description, type, and degree. For the calculator to execute with the highest accuracy, prior data observation is necessary. Below is a sample set of columns to observe. Notice how `Offence Type Desc` has inconsistent values. Some values in the `Offence Type Desc` are the type and class, others are only the type, and many have null values. At the same time, the `Offense Desc` and `Offense Degree Desc` is completely null. This is very important observation when choosing which column is best fit for the `_Col_Class` attribute in the constructor.

In [10]:
test_data = pd.read_csv('TestCSV_FilesInput/Test4_CY21-CY24 Jail Data DRAFT.csv')
file_rExample = test_data.iloc[
    [0, 7, 1102, 135, 51, 8, 83]][
    ['Offence Code', 'Offence Type Desc', 'Offense Desc', 'Offense Degree Desc', 'Case Info Num', ]]
file_rExample

,Offence Code,Offence Type Desc,Offense Desc,Offense Degree Desc,Case Info Num
0,385500.0,NaN,NaN,NaN,17CVD361
7,540500.0,TRAFFIC,NaN,NaN,21CR200047
1102,444000.0,INFRACTION,NaN,NaN,20CR700184
135,102000.0,FELONY,NaN,NaN,20CR219417
51,291200.0,MISDEMEANOR,NaN,NaN,21CR200076
8,220500.0,FELONY - CLASS H,NaN,NaN,20CRS015059
83,133600.0,MISDEMEANOR - CLASS 1,NaN,NaN,13CR237218


### **Demonstration**
In this section, there will three different walk-throughs consisting of a simple version and an in detailed version of how to utilize the `cjs.DataSource()` library.

##### **Demo 1: Simplistic Code Review**
This is a more simplistic version of how the code should look if doing a simple execution of finding the highest crime including only class A results. The test data will be referred to as `test4_data`.

In [11]:
# Imports
import pandas as pd
import CJ_Services as cjs

# Initialize
all_offense = pd.read_csv('TestCSV_FilesInput/Test_OffenseCodes.csv')
test5_data = cjs.DataSource(
    CSV_File='TestCSV_FilesInput/Test4_CY21-CY24 Jail Data DRAFT.csv',
    Col_Class='Offence Type Desc',
    Col_Code='Offence Code'
)

# Fill in offense class (Optional fill in other columns)
test5_data.Fill_BlankCol(
    File_OffenseCodes=all_offense,
    Value_Exception=['felony', 'misdemeanor', 'traffic', 'infraction']
)

# Calculate highest offense with results of class A
test5_data.Calc_HighestOffense(
    Group_By='Case Info Num',
    Find_Class='A'
)

# Export CSV
test5_data.Export_File(
    Folder_Path='TestCSV_FilesOutput/',
    File_Name='test5'
)

Dropped 25196 rows where 'Code (4)' was missing.
Exported in "TestCSV_FilesOutput/" folder as "PROCESSED_test5.csv"



##### **Demo 2: Detailed Walk-Through**
For this demonstration, two variables of the same data will be used to display the different format alternatives. These variables will be called `test1_data`, `test2_data`, and `test3_data`

**1.** Specified libraries must be imported to your *File.py* or *File.ipynb*. These imports are *Pandas* library and the personal *CJ_Services* library. As written below, libraries can be imported as personal phrases, however, standard practice is to use their abbreviation. 

In [12]:
import pandas as pd
import CJ_Services as cjs

**2.** Initializing *All_OffenseCodes.csv* file. This will be done using the `pd.read_csv()` method.

In [13]:
all_offense = pd.read_csv('TestCSV_FilesInput/Test_OffenseCodes.csv')

**3.** Initializing the *Test1_CY21-CY24 Jail Data DRAFT.csv* file. This spreadsheet will utilize the newly created class and initiated like this, `cjs.DataSource()`. Keep in mind that each attribute required or optional must be written and equaled to their perspective values. Python is case sensitive to indentions, but other formatting is personal preference and below is standard practice.

**Note:** Be aware spelling errors may occur when specifying each column, as this portion of the code is case sensitive. 

In [14]:
# Option 1: w/ PrimKey
test1_data = cjs.DataSource(
    CSV_File='TestCSV_FilesInput/Test1_CY21-CY24 Jail Data DRAFT.csv',
    Col_PrimKey='Case Info Num',
    Col_Class='Offence Type Desc',
    Col_Code='Offence Code'
)
# Option 2: w/out PrimKey
test2_data = cjs.DataSource(
    CSV_File='TestCSV_FilesInput/Test2_CY21-CY24 Jail Data DRAFT.csv',
    Col_Class='Offence Type Desc',
    Col_Code='Offence Code'
)

# Option 3: w/ pd.DataFrame() file
test3 = pd.read_csv('TestCSV_FilesInput/Test3_CY21-CY24 Jail Data DRAFT.csv')
test3_data = cjs.DataSource(
    CSV_File=test3,
    Col_Class='Offence Type Desc',
    Col_Code='Offence Code'
)

**4A.** As stated before, the `Offence Type Desc` has a variety of values that contain an offense degree and ones that don't. This means prior to using the `cjs.Fill_BlankCol()` method, the `Value_Exception` parameter has to be utilized to prevent the unwanted values to override the method. This argument must be a singular string or a list of strings, it also can be written in the parameter or outside of it, dependent on number of exception cases. Stated prior, the `Value_Exception` argument is not case sensitive. Also, the automatic assumption is that the function is filling in the dataset's offense classes if no other parameters are written. 

In [15]:
# Option 1: Exception outside of method
exceptions = ['felony', 'MISDEMEANOR']
test1_data.Fill_BlankCol(
    File_OffenseCodes=all_offense,
    Value_Exception=exceptions
)

# Option 2: Exception inside the method
test2_data.Fill_BlankCol(
    File_OffenseCodes=all_offense,
    Value_Exception=['FELONY', 'MISDEMEANOR']
)

# Option 3: Fill specifying column names, not necessary for Class Column
test3_data.Fill_BlankCol(
    File_OffenseCodes=all_offense, 
    File_colNull='Offense Description',
    Data_colNull='Offense Desc',
    Value_Exception=['Felony', 'Misdemeanor', 'Infraction', 'Traffick']
)

Dropped 25196 rows where 'Code (4)' was missing.
Dropped 25196 rows where 'Code (4)' was missing.
Dropped 25196 rows where 'Code (4)' was missing.


**4B.** The second portion of this function is calling another function to look at all the dropped data. Again, this is to check if there are issues with the `cjs.Fill_BlankCol()` and it dropping the wrong values. The main idea of this is that if there is a high percentage of dropped data, then it is necessary to verfiy that the data being dropped is due to the fact that `_Col_Code` values are missing or invalid.

**Note:** Users must assign a variable to calling the method, in order to view dropped rows in a dataset.

In [16]:
test2_dropped = test2_data.Get_Dropped()

**4C.** The after checking the drooped data for any irregularity, utilizing the previous function, it can fill in any column related to offense. For these datasets, the offense description is missing. Due to this, all permanent operations must be stated prior to non-permanent operations. The only non-permanent operation currently is the calculator method. Below is a demonstration of how users were to fill in other null columns. When performing `cjs.Fill_BlankCol()` method several times, the automatic drop text will continue to inform how many more rows have been dropped.

**Note:** Rows dropped is zero because previously, this is because the function has already ran once, dropping all the NULL rows. So any future attempts of `cjs.Fill_BlankCol()` will state zero as the number of rows dropped.

In [17]:
test2_data.Fill_BlankCol(
    File_OffenseCodes=all_offense,
    File_colNull='Offense Description',
    Data_colNull='Offense Desc'
)

Dropped 0 rows where 'Code (4)' was missing.


**5A.** After making sure all the offense degrees are accounted for, users can use the `cjs.Calc_HighestOffense()` method. The `test4_data` will demonstrate finding the highest offense of the entire dataset, grouped by the primary key, which is case number. On the other hand, `test5_data` will demonstrate calculating the highest offense of a specific class. Since this data was not initialized with a primary key column, users will have to specify how to group the data. For comparison, the case number will be the grouping metric.

In [18]:
# Option 1: no attribute declaration
test1_data.Calc_HighestOffense()

# Option 2: grouped_by
test2_data.Calc_HighestOffense(
    Group_By='Case Info Num', 
    Find_Class='c')

**5B**. To view or do more operations in python, the `cjs.Get_Data()` method will have to be used as follows. This method allows for specification of pre- or post-data, based on if users utilized a non-permanent operation. In the *Option 1* is a prime example of how it will return the calculated data, stated as the first line of the output. This is because in the previous step, *5A*, the `cjs.Calc_HighestOffense()` had been initiated initializing the `_newData` variable in the constructor. To pull the original data without specifying, it'll only work if a non-permanent method has not been initialized.

In [ ]:
# --- Calculated Data ---
# Option 1: no parameter specification
test2_calc = test2_data.Get_Data()

# Option 2: parameter specified
test2_calc = test2_data.Get_Data('calculated data')

# --- Original Data ---
# Option 3: no parameter specification and no data in the _newData variable
test3_orig = test3_data.Get_Data()

# Option 4: parameter specified
test3_orig = test3_data.Get_Data('original data')

# --- No Dataset Return --- 
# Option 5: no data in the _newData variable
test3_none = test3_data.Get_Data('calculated data')

Returning Calculated Data
Returning Calculated Data
Returning Original Data
Returning Original Data
Data has not been calculated


**6.** Lastly, after all operations have been completed to user needs, the `pandas.DataFrame()` or `cjs.DataSource` can be exported. If the dataset was initialized into a new variable, like in step 5B, export will need to be manual using `pandas.to_csv()`. If done this way, there will not be a printed out statement of where the file is. Otherwise, the dataset is still it's original `DataSource` type and can be exported using `cjs.Export_File()` method. This method does have an attribute where users can input a folder to where they would like their CSV file to be. Make sure you thoroughly understand how to format the string for `Folder_Path=` as it is different based on the type of file python code is executing and the location of folder relative to the file. To prevent confusion, a statement will print out as to where the file is, if folder is specified, and what the file name is. If folder is not specified, automatic assumption must be that file is in the same folder as the python script.

**Note:** Option 4 does not have a print statement of where and what the file name is due to the fact that users have to manually write in code where the file goes. Rather than the `cjs.Export_File()` function which were in Option 1, the execution does not require to write a path.


In [ ]:
# Option 1: no parameter specification
test1_data.Export_File()

# Option 2: parameter specified
test2_data.Export_File(Folder_Path='TestCSV_FilesOutput')

# Option 3: w/ file name
test3_data.Export_File(
    Folder_Path='TestCSV_FilesOutput/',
    File_Name='test3'
)

# Option 4: pandas.DataFrame() method, if dataset is no longer a DataSource type
test2_calc.to_csv(
    f"TestCSV_FilesOutput/PROCESSED_Test5_CY21-CY24 Jail Data Draft.csv", 
    index=False, 
    encoding='utf-8'
)


Exported as "_PROCESSED_Test1_CY21-CY24 Jail Data DRAFT.csv"

Exported in "TestCSV_FilesOutput/" folder as "PROCESSED_Test2_CY21-CY24 Jail Data DRAFT.csv"

Exported in "TestCSV_FilesOutput/" folder as "PROCESSED_test3.csv"



##### **Demo 3: Results Comparison**
To better understand the calculator, below is a results comparison of before and after each method once it has been run. The testing data variable will be referred to as `test5_data`. In this results comparison, traffic and infraction keywords will be put as part of the `Value_Exception=` cases. When viewing the results, know that not all traffic and infraction cases have a class description or an offense description, leading to NULL value appearance.

In [21]:
# Initialize
test5_data = cjs.DataSource(
    CSV_File='TestCSV_FilesInput/Test4_CY21-CY24 Jail Data DRAFT.csv',
    Col_Class='Offence Type Desc',
    Col_Code='Offence Code'
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Offence Code', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Returning Original Data


,Case Info Num,Offence Code,Offence Type Desc,Offense Desc
0,17CVD361,385500.0,NaN,NaN
1,19CR053275,132500.0,NaN,NaN
2,20CRS015942,236500.0,NaN,NaN
3,20CRS015941,232100.0,NaN,NaN
4,HLD-169813,NaN,NaN,NaN
5,20CRS015029,102800.0,NaN,NaN
6,20CRS015031,138900.0,NaN,NaN
7,21CR200047,540500.0,TRAFFIC,NaN
8,20CRS015059,220500.0,FELONY - CLASS H,NaN
9,20CRS015030,132300.0,NaN,NaN


In [22]:
# --- Test 5: Backfilled Class ---
test5_data.Fill_BlankCol(
    File_OffenseCodes=all_offense,
    Value_Exception=['felony', 'misdemeanor', 'traffic', 'infraction']
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Code (4)', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Dropped 25196 rows where 'Code (4)' was missing.
Returning Original Data


,Case Info Num,Code (4),Offence Type Desc,Offense Desc
0,17CVD361,3855.0,1/2,NaN
1,19CR053275,1325.0,F,NaN
2,20CRS015942,2365.0,I,NaN
3,20CRS015941,2321.0,H,NaN
4,20CRS015029,1028.0,E,NaN
5,20CRS015031,1389.0,A1,NaN
6,21CR200047,5405.0,,NaN
7,20CRS015059,2205.0,FELONY - CLASS H,NaN
8,20CRS015030,1323.0,H,NaN
9,21CR200125,5310.0,2,NaN


In [23]:
# --- Test 5 Backfilled Description ---
test5_data.Fill_BlankCol(
    File_OffenseCodes=all_offense,
    File_colNull='Offense Description',
    Data_colNull='Offense Desc'
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Code (4)', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Dropped 0 rows where 'Code (4)' was missing.
Returning Original Data


,Case Info Num,Code (4),Offence Type Desc,Offense Desc
0,17CVD361,3855.0,1/2,NONSUPPORT CHILD
1,19CR053275,1325.0,F,ASSAULT SERIOUS BODILY INJURY
2,20CRS015942,2365.0,I,CONSPIRE COMMIT FEL LARCENY
3,20CRS015941,2321.0,H,FELONY LARCENY
4,20CRS015029,1028.0,E,SECOND DEGREE KIDNAPPING
5,20CRS015031,1389.0,A1,ASSAULT ON A FEMALE
6,21CR200047,5405.0,,DRIVING WHILE IMPAIRED
7,20CRS015059,2205.0,FELONY - CLASS H,BREAK/ENTER TERRORIZE/INJURE
8,20CRS015030,1323.0,H,ASSAULT BY STRANGULATION
9,21CR200125,5310.0,2,RESISTING PUBLIC OFFICER


In [24]:
# --- Test 5 Check Dropped Data ---
test5_drop = test5_data.Get_Dropped()
test5_sub = test5_drop[['Case Info Num', 'Code (4)', 'Offence Type Desc_original', 'Offense Desc']].head(10)
test5_sub

,Case Info Num,Code (4),Offence Type Desc_original,Offense Desc
4,HLD-169813,NaN,NaN,NaN
18,HLD-170092,NaN,NaN,NaN
37,HLD-169810,NaN,NaN,NaN
42,HLD-169815,NaN,NaN,NaN
49,HLD-169821,NaN,NaN,NaN
55,NaN,NaN,NaN,NaN
63,HLD-169811,NaN,NaN,NaN
64,HLD-169814,NaN,NaN,NaN
96,HLD-169820,NaN,NaN,NaN
109,HLD-169818,NaN,NaN,NaN


In [25]:
# --- Test 5: Calculate Highest Offense ---
test5_data.Calc_HighestOffense(
    Group_By='Case Info Num'
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Offence Code', 'Code (4)', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Returning Calculated Data


,Case Info Num,Offence Code,Code (4),Offence Type Desc,Offense Desc
64389,#CONTEMPT,502800.0,5028.0,Blank,CRIMINAL CONTEMPT
53476,#GOV'S WARRANT,990101.0,9901.0,Blank,EXTRADITION/FUGITIVE OTH STATE
53507,#GOV'SWARRANT,990101.0,9901.0,Blank,EXTRADITION/FUGITIVE OTH STATE
62104,.,138800.0,1388.0,A1,ASSAULT WITH A DEADLY WEAPON
148218,000000,999900.0,9999.0,??,OTHER - FREE TEXT
121578,00CR000000,220600.0,2206.0,I,POSSESSION OF BURGLARY TOOLS
34689,00CR000001,122000.0,1220.0,G,COMMON LAW ROBBERY
34687,00CR000002,138902.0,1389.0,A1,ASSAULT ON A FEMALE
40569,00CR029236,503200.0,5032.0,Blank,FELONY PROBATION VIOLATION
197159,00CR053542,232200.0,2322.0,1,MISDEMEANOR LARCENY


In [26]:
# --- Test 5: Calculate Highest Offense by Class A ---
test5_data.Calc_HighestOffense(
    Group_By='Case Info Num',
    Find_Class='A'
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Offence Code', 'Code (4)', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Returning Calculated Data


,Case Info Num,Offence Code,Code (4),Offence Type Desc,Offense Desc
80661,06CR053940,342600.0,3426.0,A/C/D/E,CS HIRE/USE MINOR<=13 DEF>=21
147327,10CR225973,93000.0,930.0,A/B1/B2,MURDER
63794,12CR249824,93500.0,935.0,A,MURDER
112931,17CR024118,503200.0,5032.0,FELONY - CLASS A,FELONY PROBATION VIOLATION
169206,17CR203745,93500.0,935.0,A,FIRST DEGREE MURDER
53301,17CRS208594,93500.0,935.0,A,FIRST DEGREE MURDER
26286,17CRS246759,93500.0,935.0,A,FIRST DEGREE MURDER
62049,18CR216550,93000.0,930.0,A/B1/B2,MURDER
130897,18CR225315,93500.0,935.0,A,FIRST DEGREE MURDER
75809,18CR231736,93500.0,935.0,A,FIRST DEGREE MURDER


In [27]:
# --- Test 4: Calculate Highest Offense by Class B1 ---
test5_data.Calc_HighestOffense(
    Group_By='Case Info Num',
    Find_Class='B1'
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Offence Code', 'Code (4)', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Returning Calculated Data


,Case Info Num,Offence Code,Code (4),Offence Type Desc,Offense Desc
71601,03CRS241674,110300.0,1103.0,B1,FIRST DEGREE RAPE
75808,09CR244949,113700.0,1137.0,B1,STAT RAPE/SEX OFFN DEF >=6YR
30844,09CR245484,113700.0,1137.0,B1,STAT RAPE/SEX OFFN DEF >=6YR
147327,10CR225973,93000.0,930.0,A/B1/B2,MURDER
26547,11CR206812,111602.0,1116.0,B1,SEX OFFENSE-1ST DEGREE-VICTIM UNDER 13
59719,12CR229279,113202.0,1132.0,B1,FIRST DEGREE SEXUAL OFFENSE
59726,12CR229285,113202.0,1132.0,B1,FIRST DEGREE SEXUAL OFFENSE
42939,13CR209574,111600.0,1116.0,B1,FIRST DEGREE SEX OFFENSE CHILD <13
143043,13CR240573,110300.0,1103.0,B1,FIRST DEGREE RAPE
143045,13CR240574,113200.0,1132.0,B1,FIRST DEGREE SEXUAL OFFENSE


In [28]:
# --- Test 4: Calculate Highest Offense by Class B2 ---
test5_data.Calc_HighestOffense(
    Group_By='Case Info Num',
    Find_Class='B2'
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Offence Code', 'Code (4)', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Returning Calculated Data


,Case Info Num,Offence Code,Code (4),Offence Type Desc,Offense Desc
15935,08CR200886,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
187258,08CR205184,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
36856,08CR215236,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
174764,08CR225823,991801.0,9918.0,B2/C/D/E/F/G/H/I,FELONY CONSPIRACY
98026,08CR250737,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
95406,08CR252199,991801.0,9918.0,B2/C/D/E/F/G/H/I,FELONY CONSPIRACY
129276,09CR210534,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
115000,09CR237113,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
123476,09CR246180,95100.0,951.0,B2,ATTEMPTED FIRST DEGREE MURDER
123473,09CR246181,95100.0,951.0,B2,ATTEMPTED FIRST DEGREE MURDER


In [29]:
# --- Test 4: Calculate Highest Offense by Class C ---
test5_data.Calc_HighestOffense(
    Group_By='Case Info Num',
    Find_Class='C'
)
test5_orig = test5_data.Get_Data()
test5_sub = test5_orig[['Case Info Num', 'Offence Code', 'Code (4)', 'Offence Type Desc', 'Offense Desc']].head(10)
test5_sub

Returning Calculated Data


,Case Info Num,Offence Code,Code (4),Offence Type Desc,Offense Desc
66108,05CR251149,112201.0,1122.0,C,RAPE-2ND DEGREE BY FORCE AGAINST WILL OF
157903,05CRS204061,112201.0,1122.0,C,SECOND DEGREE RAPE
80661,06CR053940,342600.0,3426.0,A/C/D/E,CS HIRE/USE MINOR<=13 DEF>=21
30854,06CR230434,268600.0,2686.0,C/H,AID AND ABET - FALSE PRETENSE
40150,06CR235512,102600.0,1026.0,C,KIDNAPPING - FIRST DEGREE
40152,06CR235515,102600.0,1026.0,C,KIDNAPPING - FIRST DEGREE
80261,07CR236678,102600.0,1026.0,C,KIDNAPPING - FIRST DEGREE
15935,08CR200886,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
187258,08CR205184,991801.0,9918.0,B2/C/D/E/F/G/H/I,CONSPIRACY - FELONY
68263,08CR213489,353100.0,3531.0,C/E/F,C/S-SCH I- TRAFFICKING IN HEROIN
